# Large Enterprise vs. Mittelstand — AI Language Benchmark
**Bachelor Thesis — Comparative Text Mining**

Counts every occurrence of **"AI"** (whole-word) and **"artificial intelligence"** across 17 annual reports,  
normalised by document length. Compares 15 Mittelstand firms against 2 large-enterprise benchmarks (SAP, Deutsche Telekom).

In [ ]:
import pdfplumber
import pandas as pd
import re
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

REPORTS_DIR = Path("reports")

# ── Group definitions ─────────────────────────────────────────────────────────
MITTELSTAND = [
    "ATOSS.pdf", "Bechtle.pdf", "Nemetscheck_Group.pdf", "Durr.pdf",
    "Secunet.pdf", "Jenoptik.pdf", "TRUMPF.pdf", "Heidelberg_Druckmaschinen.pdf",
    "Rational_AG.pdf", "STRATEC_SE.pdf", "Krones.pdf", "SUSS MicroTec SE.pdf",
    "Deutz_AG.pdf", "Dermapharm_SE.pdf", "Wacker_Neuson.pdf",
]
LARGE_ENTERPRISE = ["SAP.pdf", "Deutsche_Telekom.pdf"]

# ── Colour palette ────────────────────────────────────────────────────────────
C_LARGE       = "#162447"   # dark navy  — Large Enterprise
C_MITTELSTAND = "#E8735A"   # coral red  — Mittelstand (default bar fill)
C_STRATEGIC   = "#C8102E"   # red        — Strategic-Dominant
C_OPERATIONAL = "#2D7D46"   # green      — Operational-Dominant
C_BALANCED    = "#D4A017"   # amber      — Balanced
C_SILENT      = "#9CA3AF"   # grey       — AI-Silent

CLASS_COLOR = {
    "Strategic-Dominant":   C_STRATEGIC,
    "Operational-Dominant": C_OPERATIONAL,
    "Balanced":             C_BALANCED,
    "AI-Silent":            C_SILENT,
    "Large Enterprise":     C_LARGE,
}

In [ ]:
def normalize_text(text: str) -> str:
    text = re.sub(r'-\n\s*', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def extract_text(pdf_path: Path) -> str:
    pages = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            t = page.extract_text()
            if t:
                pages.append(t)
    return normalize_text("\n".join(pages))


def count_ai(text: str) -> dict:
    """Count whole-word AI and 'artificial intelligence' mentions."""
    t = text.lower()
    n_ai     = len(re.findall(r'\bai\b', t))
    n_full   = len(re.findall(r'\bartificial intelligence\b', t))
    total_w  = max(len(t.split()), 1)
    total_m  = n_ai + n_full
    density  = round((total_m / total_w) * 10_000, 3)
    return {"ai_mentions": n_ai, "artif_intel_mentions": n_full,
            "total_mentions": total_m, "total_words": total_w, "per_10k": density}


def label_from_filename(name: str) -> str:
    stem = Path(name).stem
    for tok in ("AG", "SE", "SAP"):
        stem = re.sub(rf'(?i)\b{tok}\b', tok, stem)
    stem = re.sub(r'[_-]+', ' ', stem).strip()
    return ' '.join(w if w.isupper() else w.capitalize() for w in stem.split())

In [ ]:
# Load previous Mittelstand classifications
prev = pd.read_csv("ai_gap_results.csv")[["company", "classification"]]
classif_map = dict(zip(prev["company"], prev["classification"]))

rows = []
all_files = [(f, "Mittelstand") for f in MITTELSTAND] + [(f, "Large Enterprise") for f in LARGE_ENTERPRISE]

print(f"{'Company':<35} {'Group':<18} {'AI':>5} {'ArtifIntel':>10} {'Total':>7} {'Words':>8} {'Per10k':>7}")
print("-" * 95)

for fname, group in all_files:
    pdf_path = REPORTS_DIR / fname
    company  = label_from_filename(fname)
    text     = extract_text(pdf_path)
    counts   = count_ai(text)
    classif  = classif_map.get(company, "Large Enterprise" if group == "Large Enterprise" else "AI-Silent")

    print(f"{company:<35} {group:<18} {counts['ai_mentions']:5d} {counts['artif_intel_mentions']:10d} "
          f"{counts['total_mentions']:7d} {counts['total_words']:8d} {counts['per_10k']:7.2f}")

    rows.append({"company": company, "group": group, "classification": classif, **counts})

df = pd.DataFrame(rows).sort_values("total_mentions", ascending=False).reset_index(drop=True)
df.to_csv("benchmark_results.csv", index=False)
print("\nSaved: benchmark_results.csv")

In [ ]:
# ── Console summary table ─────────────────────────────────────────────────────
print(f"{'Company':<35} {'Group':<18} {'Total Mentions':>14} {'Per 10k Words':>14} {'Classification'}")
print("-" * 105)
for _, r in df.iterrows():
    print(f"{r.company:<35} {r.group:<18} {r.total_mentions:14d} {r.per_10k:14.2f}   {r.classification}")

## Charts

In [ ]:
# ── Chart 1: Absolute AI mentions horizontal bar chart ───────────────────────
g1_avg = df[df.group == "Mittelstand"]["total_mentions"].mean()

fig, ax = plt.subplots(figsize=(11, 8))
colors  = [C_LARGE if r.group == "Large Enterprise" else C_MITTELSTAND for _, r in df.iterrows()]
bars    = ax.barh(df["company"], df["total_mentions"], color=colors, edgecolor="white", linewidth=0.5)

ax.axvline(g1_avg, color=C_STRATEGIC, linestyle="--", linewidth=1.4,
           label=f"Mittelstand avg ({g1_avg:.0f})")
ax.bar_label(bars, labels=[str(v) for v in df["total_mentions"]], padding=4, fontsize=9)
ax.set_xlabel("Total AI Mentions")
ax.set_title("Total AI Mentions in Annual Reports (2024)", fontweight="bold", fontsize=13)
ax.invert_yaxis()

legend_handles = [
    mpatches.Patch(color=C_LARGE,       label="Large Enterprise (SAP, Deutsche Telekom)"),
    mpatches.Patch(color=C_MITTELSTAND, label="Mittelstand"),
    plt.Line2D([0], [0], color=C_STRATEGIC, linestyle="--", label=f"Mittelstand avg ({g1_avg:.0f})"),
]
ax.legend(handles=legend_handles, fontsize=9, loc="lower right")
ax.set_xlim(0, df["total_mentions"].max() * 1.18)
sns_despine = lambda: [ax.spines[s].set_visible(False) for s in ("top", "right")]
sns_despine()
plt.tight_layout()
plt.savefig("chart_mentions_absolute.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: chart_mentions_absolute.png")

In [ ]:
# ── Chart 2: Density bar chart (per 10,000 words) ────────────────────────────
df_d    = df.sort_values("per_10k", ascending=False).reset_index(drop=True)
g1_avg_d = df_d[df_d.group == "Mittelstand"]["per_10k"].mean()

fig, ax = plt.subplots(figsize=(11, 8))
colors  = [C_LARGE if r.group == "Large Enterprise" else C_MITTELSTAND for _, r in df_d.iterrows()]
bars    = ax.barh(df_d["company"], df_d["per_10k"], color=colors, edgecolor="white", linewidth=0.5)

ax.axvline(g1_avg_d, color=C_STRATEGIC, linestyle="--", linewidth=1.4,
           label=f"Mittelstand avg ({g1_avg_d:.2f})")
ax.bar_label(bars, labels=[f"{v:.2f}" for v in df_d["per_10k"]], padding=4, fontsize=9)
ax.set_xlabel("AI Mentions per 10,000 Words")
ax.set_title("AI Mention Density per 10,000 Words", fontweight="bold", fontsize=13)
ax.invert_yaxis()

legend_handles = [
    mpatches.Patch(color=C_LARGE,       label="Large Enterprise (SAP, Deutsche Telekom)"),
    mpatches.Patch(color=C_MITTELSTAND, label="Mittelstand"),
    plt.Line2D([0], [0], color=C_STRATEGIC, linestyle="--", label=f"Mittelstand avg ({g1_avg_d:.2f}/10k)"),
]
ax.legend(handles=legend_handles, fontsize=9, loc="lower right")
ax.set_xlim(0, df_d["per_10k"].max() * 1.18)
[ax.spines[s].set_visible(False) for s in ("top", "right")]
plt.tight_layout()
plt.savefig("chart_mentions_density.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: chart_mentions_density.png")

In [ ]:
# ── Chart 3: Group comparison — avg absolute vs avg density ──────────────────
g1 = df[df.group == "Mittelstand"]
g2 = df[df.group == "Large Enterprise"]

avg_abs  = [g1["total_mentions"].mean(), g2["total_mentions"].mean()]
avg_dens = [g1["per_10k"].mean(),        g2["per_10k"].mean()]
labels   = ["Mittelstand\n(n=15)", "Large Enterprise\n(n=2)"]
grp_colors = [C_MITTELSTAND, C_LARGE]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
fig.suptitle("Large Enterprise vs. Mittelstand — AI Language Volume",
             fontweight="bold", fontsize=13)

# Left: average absolute mentions
b1 = ax1.bar(labels, avg_abs, color=grp_colors, edgecolor="white", width=0.5)
ax1.bar_label(b1, fmt="%.0f", padding=4, fontsize=11, fontweight="bold")
ax1.set_ylabel("Average total AI mentions")
ax1.set_title("Average Absolute Mentions", fontsize=11)
ax1.set_ylim(0, max(avg_abs) * 1.25)
[ax1.spines[s].set_visible(False) for s in ("top", "right")]

# Right: average density per 10k
b2 = ax2.bar(labels, avg_dens, color=grp_colors, edgecolor="white", width=0.5)
ax2.bar_label(b2, fmt="%.2f", padding=4, fontsize=11, fontweight="bold")
ax2.set_ylabel("Average AI mentions per 10,000 words")
ax2.set_title("Average Density (per 10k words)", fontsize=11)
ax2.set_ylim(0, max(avg_dens) * 1.25)
[ax2.spines[s].set_visible(False) for s in ("top", "right")]

plt.tight_layout()
plt.savefig("chart_group_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: chart_group_comparison.png")

In [ ]:
# ── Chart 4: Dot plot — density distribution, all 17 companies ───────────────
df_dot = df.sort_values("per_10k", ascending=True).reset_index(drop=True)
g1_avg_d = df[df.group == "Mittelstand"]["per_10k"].mean()
g2_avg_d = df[df.group == "Large Enterprise"]["per_10k"].mean()

fig, ax = plt.subplots(figsize=(13, 7))

for i, (_, row) in enumerate(df_dot.iterrows()):
    color  = CLASS_COLOR.get(row["classification"], C_SILENT)
    if row["group"] == "Large Enterprise":
        ax.scatter(row["per_10k"], i, marker="*", s=420, color=C_LARGE,
                   zorder=4, linewidths=0.5, edgecolors="white")
    else:
        ax.scatter(row["per_10k"], i, marker="o", s=120, color=color,
                   zorder=4, linewidths=0.5, edgecolors="white")
    # Faint horizontal guide line
    ax.axhline(i, color="#e5e5e5", linewidth=0.6, zorder=1)

# Average lines
ax.axvline(g1_avg_d, color=C_STRATEGIC, linestyle="--", linewidth=1.4,
           label=f"Mittelstand avg ({g1_avg_d:.2f}/10k)", zorder=3)
ax.axvline(g2_avg_d, color=C_LARGE,     linestyle="--", linewidth=1.4,
           label=f"Large Enterprise avg ({g2_avg_d:.2f}/10k)", zorder=3)

ax.set_yticks(range(len(df_dot)))
ax.set_yticklabels(df_dot["company"], fontsize=9.5)
ax.set_xlabel("AI Mentions per 10,000 Words", fontsize=11)
ax.set_title("AI Language Density Distribution — All 17 Companies",
             fontweight="bold", fontsize=13)

# Legend
legend_handles = [
    plt.scatter([], [], marker="*", s=200, color=C_LARGE,       label="Large Enterprise"),
    plt.scatter([], [], marker="o", s=80,  color=C_STRATEGIC,   label="Strategic-Dominant"),
    plt.scatter([], [], marker="o", s=80,  color=C_OPERATIONAL, label="Operational-Dominant"),
    plt.scatter([], [], marker="o", s=80,  color=C_BALANCED,    label="Balanced"),
    plt.scatter([], [], marker="o", s=80,  color=C_SILENT,      label="AI-Silent"),
    plt.Line2D([0],[0], color=C_STRATEGIC, linestyle="--", label=f"Mittelstand avg ({g1_avg_d:.2f})"),
    plt.Line2D([0],[0], color=C_LARGE,     linestyle="--", label=f"Large Enterprise avg ({g2_avg_d:.2f})"),
]
ax.legend(handles=legend_handles, fontsize=8.5, loc="lower right", framealpha=0.9)
[ax.spines[s].set_visible(False) for s in ("top", "right")]
plt.tight_layout()
plt.savefig("chart_dotplot.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: chart_dotplot.png")